In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.fractional_OU import fit_cointegrated_pairs_fractional_ou


# 03 Fractional OU Estimation

Fit the fractional OU model to all retained formation spreads. The next module applies anti-persistence and numerical-stability eligibility rules.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load selected formation spreads

No out-of-sample prices enter estimation.


In [ ]:
selected = pd.read_parquet("cointegrated_pairs.parquet")
formation_spreads = pd.read_parquet("formation_spreads.parquet")
spreads = {(r.dependent, r.independent): formation_spreads[r.pair] for r in selected.itertuples()}
display(selected[["pair", "alpha", "beta"]].head())


## 3. Estimate parameters

The continuous-time stationary-variance estimate and daily Euler simulator remain model approximations. Fit errors and boundary H estimates are recorded.


In [ ]:
fou, fit_audit = fit_cointegrated_pairs_fractional_ou(spreads, selected, return_audit=True)
fit_audit.to_parquet("fou_fit_audit.parquet")
if fou.empty:
    raise ValueError("No valid fOU fits. Inspect fou_fit_audit.")
fou.to_parquet("fractional_ou_parameters.parquet")
display(fou.head(10))
display(fit_audit.head(10))


## 4. Hurst summary

The summary refers to exactly the fitted population saved above.


In [ ]:
hurst_summary = fou.hurst.describe()
hurst_summary.to_frame("hurst").to_parquet("hurst_summary.parquet")
display(hurst_summary)
fou.hurst.hist(bins=25, figsize=(8, 3))
plt.title("Formation Hurst estimates")
plt.xlabel("H")
plt.show()
